In [1]:
!pip install transformers datasets seqeval sentencepiece -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import ast, re, json, warnings, random
import numpy as np
import pandas as pd
from collections import Counter
from datasets import Dataset
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import DataLoader
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("CUDA:", torch.cuda.is_available())


Device: cuda
CUDA: True


## 1. Load & Preprocess Dataset

In [3]:
df = pd.read_csv("https://docs.google.com/spreadsheets/d/1KXcA0PPOpygla1inEfnTc10FND6DFNL8p8vpQKAX6Tw/export?format=csv")
print("Shape:", df.shape)
df.head(3)


Shape: (4000, 6)


,parent_asin,sentence_id,sentence_text,rating,triplets,category_name
0,B07DGRVTWF,3,basically a poor implementation of the alexa p...,2.0,"[[""alexa platform"", ""poor implementation"", 0]]",electronics_p2
1,B07BFPJ6VX,4,support through direct messaging was great no ...,5.0,"[[""support through direct messaging"", ""great"",...",electronics_p2
2,B0B1PYNH8X,1,for echo show 5 i have the 2nd generation echo...,5.0,[],electronics_p2


In [4]:
SENTIMENT_MAP     = {0: "negative", 1: "neutral", 2: "positive",
                     "0": "negative", "1": "neutral", "2": "positive"}
SENTIMENT_MAP_REV = {"negative": 0, "neutral": 1, "positive": 2}

def safe_parse(x):
    """Parse triplet string: hỗ trợ cả list-of-list và list-of-dict, int hoặc str sentiment."""
    try:
        raw = str(x).strip().replace('\"\"', '\"')
        v   = ast.literal_eval(raw)
        if not isinstance(v, list):
            return []
        out = []
        for t in v:
            if isinstance(t, (list, tuple)) and len(t) == 3:
                asp, opi, sen = t
                out.append([str(asp).strip(), str(opi).strip(),
                             SENTIMENT_MAP.get(int(sen) if isinstance(sen, (int,float)) else sen, "neutral")])
            elif isinstance(t, dict):
                out.append([
                    str(t.get("aspect",  "")).strip(),
                    str(t.get("opinion", "")).strip(),
                    SENTIMENT_MAP.get(t.get("sentiment", 1), "neutral"),
                ])
        return [t for t in out if t[0] and t[1]]
    except:
        return []

def triplets_to_target(triplets):
    """[[asp, opi, senti_str], ...] → '(asp, opi, senti) ; ...'"""
    parts = []
    for t in triplets:
        if len(t) == 3:
            parts.append(f"({t[0].strip()}, {t[1].strip()}, {t[2].strip()})")
    return " ; ".join(parts) if parts else "none"

df["triplets"] = df["triplets"].apply(safe_parse)
df = df[df["triplets"].apply(len) > 0].reset_index(drop=True)
df["target"]   = df["triplets"].apply(triplets_to_target)
df["input"]    = "extract triplets: " + df["sentence_text"].astype(str)

print(f"Filtered dataset: {len(df)} rows")

# Phân phối sentiment
all_s = [t[2] for trips in df["triplets"] for t in trips]
print("Sentiment dist:", Counter(all_s))


Filtered dataset: 2269 rows
Sentiment dist: Counter({'positive': 2481, 'negative': 1185, 'neutral': 173})


## 2. Train / Test Split

In [5]:
train_df, test_df = train_test_split(
    df, test_size=0.1, random_state=SEED, shuffle=True
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")


Train: 2042 | Test: 227


## 3. Data Augmentation – Cân bằng minority class

`neutral` thường chiếm rất ít → upsample bằng cách tạo thêm câu đảo thứ tự triplet và random deletion từ câu gốc.


In [6]:
def random_deletion(text: str, p: float = 0.15) -> str:
    words = text.split()
    if len(words) <= 3:
        return text
    new_words = [w for w in words if random.random() > p]
    return " ".join(new_words) if new_words else text

# Đếm sentiment để tìm minority
senti_counts = Counter(t[2] for trips in train_df["triplets"] for t in trips)
print("Before aug:", senti_counts)

# Upsample neutral (label=1) bằng cách tạo bản augmented
neutral_rows = train_df[
    train_df["triplets"].apply(lambda ts: any(t[2]=="neutral" for t in ts))
].copy()

aug_rows = []
# 3× upsample neutral rows với random deletion
for _ in range(3):
    for _, row in neutral_rows.iterrows():
        new_sent = random_deletion(row["sentence_text"], p=0.15)
        new_input  = "extract triplets: " + new_sent
        aug_rows.append({**row.to_dict(),
                         "input": new_input,
                         "sentence_text": new_sent})

aug_df   = pd.DataFrame(aug_rows)
train_df = pd.concat([train_df, aug_df], ignore_index=True).sample(
    frac=1, random_state=SEED).reset_index(drop=True)
train_df["target"] = train_df["triplets"].apply(triplets_to_target)
train_df["input"]  = "extract triplets: " + train_df["sentence_text"].astype(str)

senti_counts_after = Counter(t[2] for trips in train_df["triplets"] for t in trips)
print("After  aug:", senti_counts_after)
print(f"Train size after aug: {len(train_df)}")


Before aug: Counter({'positive': 2230, 'negative': 1081, 'neutral': 156})
After  aug: Counter({'positive': 2452, 'negative': 1297, 'neutral': 624})
Train size after aug: 2444


## 4. Tokenization

In [7]:
# ── Thay đổi: dùng t5-base (giữ nguyên từ bản gốc) ─────────────────────────
MODEL_NAME     = "t5-base"
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128   # ← giảm từ 256 xuống 128: target thực tế ngắn, tránh lãng phí

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    model_inputs = tokenizer(
        examples["input"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=examples["target"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in lab]
        for lab in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs

train_dataset = Dataset.from_dict({
    "input": train_df["input"].tolist(),
    "target": train_df["target"].tolist()
}).map(preprocess, batched=True, remove_columns=["input","target"])

test_dataset = Dataset.from_dict({
    "input": test_df["input"].tolist(),
    "target": test_df["target"].tolist()
}).map(preprocess, batched=True, remove_columns=["input","target"])

print("Train dataset:", train_dataset)
print("Test dataset :", test_dataset)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2444 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

Train dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2444
})
Test dataset : Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 227
})


## 5. Load Model

In [8]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {params:,}")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Trainable params: 222,903,552


## 6. Metrics – Triplet Exact-Match F1

In [9]:
def parse_generated_triplets(text: str):
    """Parse '(asp, opi, senti) ; ...' → list of (asp, opi, senti)"""
    pattern = r"\(([^,]+),\s*([^,]+),\s*(positive|negative|neutral)\)"
    matches = re.findall(pattern, text.lower())
    return [(m[0].strip(), m[1].strip(), m[2].strip()) for m in matches]

def compute_triplet_f1(pred_list, gold_list):
    tp = fp = fn = 0
    for preds, golds in zip(pred_list, gold_list):
        ps = set(preds); gs = set(golds)
        tp += len(ps & gs)
        fp += len(ps - gs)
        fn += len(gs - ps)
    p  = tp / (tp + fp + 1e-9)
    r  = tp / (tp + fn + 1e-9)
    f1 = 2*p*r / (p + r + 1e-9)
    return {"precision": round(p,4), "recall": round(r,4), "f1": round(f1,4)}

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    pred_t = [parse_generated_triplets(p) for p in decoded_preds]
    gold_t = [parse_generated_triplets(g) for g in decoded_labels]
    return compute_triplet_f1(pred_t, gold_t)

# Sanity check
print(parse_generated_triplets("(screen, bad, negative) ; (battery, great, positive)"))


[('screen', 'bad', 'negative'), ('battery', 'great', 'positive')]


In [10]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

BATCH        = 8
ACCUM        = 4
EPOCHS       = 8
LR           = 1e-4

training_args = Seq2SeqTrainingArguments(
    output_dir                   = "./t5_absa_v2",

    learning_rate                = LR,
    lr_scheduler_type            = "cosine",
    per_device_train_batch_size  = BATCH,
    per_device_eval_batch_size   = 16,
    gradient_accumulation_steps  = ACCUM,
    num_train_epochs             = EPOCHS,
    warmup_ratio                 = 0.1,
    weight_decay                 = 0.01,
    label_smoothing_factor       = 0.1,
    predict_with_generate        = True,
    generation_max_length        = MAX_TARGET_LEN,
    generation_num_beams         = 4,
    eval_strategy                = "epoch",
    save_strategy                = "epoch",
    load_best_model_at_end       = True,
    metric_for_best_model        = "f1",
    greater_is_better            = True,
    fp16                         = torch.cuda.is_available(),
    logging_steps                = 50,
    report_to                    = "none",
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = test_dataset,
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
)

trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,17.313435,2.092729,0.412100,0.385400,0.398300
2,8.417212,1.900502,0.474700,0.479800,0.477200
3,7.771449,1.848316,0.463500,0.496000,0.479200
4,7.468490,1.825578,0.504100,0.501300,0.502700
5,7.200450,1.818044,0.468500,0.501300,0.484400
6,7.106787,1.816070,0.476400,0.517500,0.496100
7,6.983926,1.812789,0.491000,0.514800,0.502600
8,7.000895,1.812852,0.496200,0.522900,0.509200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=616, training_loss=8.335547471975351, metrics={'train_runtime': 1085.5069, 'train_samples_per_second': 18.012, 'train_steps_per_second': 0.567, 'total_flos': 2976586169057280.0, 'train_loss': 8.335547471975351, 'epoch': 8.0})

## 8. Đánh giá trên Test Set

In [11]:
results = trainer.evaluate(test_dataset)
print(f"  Precision : {results.get('eval_precision', 0):.4f}")
print(f"  Recall    : {results.get('eval_recall',    0):.4f}")
print(f"  F1 Score  : {results.get('eval_f1',        0):.4f}")
print(f"  Loss      : {results.get('eval_loss',      0):.4f}")


  Precision : 0.4962
  Recall    : 0.5229
  F1 Score  : 0.5092
  Loss      : 1.8129


In [12]:
model.eval(); model.to(device)

all_preds, all_golds = [], []
BATCH_INF = 32

inputs_all = test_df["input"].tolist()
for i in range(0, len(inputs_all), BATCH_INF):
    batch_enc = tokenizer(
        inputs_all[i:i+BATCH_INF],
        return_tensors="pt", max_length=MAX_INPUT_LEN,
        truncation=True, padding=True,
    ).to(device)
    with torch.no_grad():
        out = model.generate(
            **batch_enc,
            max_new_tokens   = MAX_TARGET_LEN,
            num_beams        = 4,
            early_stopping   = True,
        )
    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
    for text in decoded:
        all_preds.append(parse_generated_triplets(text))

for target in test_df["target"]:
    all_golds.append(parse_generated_triplets(target))

# Overall
overall = compute_triplet_f1(all_preds, all_golds)
print(f"  Overall   P={overall['precision']:.4f}  R={overall['recall']:.4f}  F1={overall['f1']:.4f}")
print()
for senti in ["positive", "negative", "neutral"]:
    fp = [[t for t in ps if t[2]==senti] for ps in all_preds]
    fg = [[t for t in gs if t[2]==senti] for gs in all_golds]
    sc = compute_triplet_f1(fp, fg)
    print(f"  [{senti:8s}] P={sc['precision']:.4f}  R={sc['recall']:.4f}  F1={sc['f1']:.4f}")


  Overall   P=0.5163  R=0.5121  F1=0.5142

  [positive] P=0.5703  R=0.5817  F1=0.5759
  [negative] P=0.3939  R=0.3786  F1=0.3861
  [neutral ] P=0.3846  R=0.2941  F1=0.3333


## 9. Demo Inference

In [13]:
def extract_triplets(sentence: str, num_beams: int = 4):
    model.eval()
    enc = tokenizer(
        "extract triplets: " + sentence,
        return_tensors="pt", max_length=MAX_INPUT_LEN, truncation=True,
    ).to(device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_TARGET_LEN,
                             num_beams=num_beams, early_stopping=True)
    generated = tokenizer.decode(out[0], skip_special_tokens=True)
    return generated, parse_generated_triplets(generated)

test_sentences = [
    "battery is great but screen is bad",
    "support through direct messaging was great no language barrier very responsive",
    "the sound quality is excellent but the build feels cheap",
    "basically a poor implementation of the alexa platform",
    "I love this keyboard, it is very comfortable and silent",
]

for sent in test_sentences:
    gen, trips = extract_triplets(sent)
    print(f"\nInput    : {sent}")
    print(f"Generated: {gen}")
    print(f"Triplets : {trips}")



Input    : battery is great but screen is bad
Generated: (battery, great, positive) ; (screen, bad, negative)
Triplets : [('battery', 'great', 'positive'), ('screen', 'bad', 'negative')]

Input    : support through direct messaging was great no language barrier very responsive
Generated: (support, great, positive) ; (support, no language barrier, positive) ; (support, responsive, positive)
Triplets : [('support', 'great', 'positive'), ('support', 'no language barrier', 'positive'), ('support', 'responsive', 'positive')]

Input    : the sound quality is excellent but the build feels cheap
Generated: (sound quality, excellent, positive) ; (build, feels cheap, negative)
Triplets : [('sound quality', 'excellent', 'positive'), ('build', 'feels cheap', 'negative')]

Input    : basically a poor implementation of the alexa platform
Generated: (implementation, poor, negative)
Triplets : [('implementation', 'poor', 'negative')]

Input    : I love this keyboard, it is very comfortable and silent

## 10. Predictions vs Ground Truth

In [16]:
n_show  = 10
correct = 0
for _, row in test_df.sample(n_show, random_state=SEED).iterrows():
    _, pred_trips = extract_triplets(row["sentence_text"])
    gold_trips    = parse_generated_triplets(row["target"])
    match = set(pred_trips) == set(gold_trips)
    if match: correct += 1
    print(f"[{'V' if match else 'X'}] {row['sentence_text']}")
    print(f"  GOLD : {gold_trips}")
    print(f"  PRED : {pred_trips}")
    print()

print(f"Exact match: {correct}/{n_show} = {correct/n_show:.1%}")


[V] great magnetic marker pencil holder set this [GENERIC_NOUN] really works well for my use in class
  GOLD : [('magnetic marker pencil holder set', 'works well', 'positive')]
  PRED : [('magnetic marker pencil holder set', 'works well', 'positive')]

[V] and the price is wonderful for the amount o received
  GOLD : [('price', 'wonderful', 'positive')]
  PRED : [('price', 'wonderful', 'positive')]

[V] great picture it was an easy app to use and the streaming was great
  GOLD : [('picture', 'great', 'positive'), ('app', 'easy', 'positive'), ('streaming', 'great', 'positive')]
  PRED : [('picture', 'great', 'positive'), ('app', 'easy', 'positive'), ('streaming', 'great', 'positive')]

[V] this pop up card with cats in it is delightful [GENERIC_NOUN] would be happy to receive itbr [GENERIC_NOUN] thats just c
  GOLD : [('pop up card', 'delightful', 'positive')]
  PRED : [('pop up card', 'delightful', 'positive')]

[X] pbs has the most amazing documentaries
  GOLD : [('pbs', 'most amazing